# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# revenue for each order
df['revenue']=df['qty'] * df['price']

#revenue for total items sold
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

#displaying the values
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Units Sold: {total_units}")
print('Across all 400 orders, the total revenue generated was $8,520.00 with a total of 783 units sold.')


Total Revenue: $8,520.00
Total Units Sold: 783
Across all 400 orders, the total revenue generated was $8,520.00 with a total of 783 units sold.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
#grouping by category and sum revenue, then sorting from highest to lowest
by_category = (df.groupby('category', as_index=False)['revenue']
               .sum()
               .sort_values(by='revenue', ascending=False)
              )
#calculating percentage share of total revenue
by_category['share_%'] = (by_category['revenue'] / df['revenue'].sum()) * 100

#formatting the %
by_category['share_%'] = by_category['share_%'].round(1)

display(by_category)


,category,revenue,share_%
1,Food,4293.0,50.4
2,Merch,1771.5,20.8
0,Drink,1554.0,18.2
3,RainGear,901.5,10.6


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
#grouping by vendor_id and calculating the average revenue and order count
vendor_averages = (
    df.groupby('vendor_id')['revenue']
    .agg(avg_revenue= 'mean', order_count = 'count')
    .sort_values(by='avg_revenue', ascending=False)
    .round({'avg_revenue':2})
)
display(vendor_averages)
print('Vendor 1 had the highest average order revenue of $22.60 while Vendor 18 had the highest order count of 108')

,avg_revenue,order_count
vendor_id,,
V-01,22.60,94
V-18,21.75,108
V-05,20.58,93
V-10,20.31,105


Vendor 1 had the highest average order revenue of $22.60 while Vendor 18 had the highest order count of 108


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# summing revenue for Merch order
merch_revenue = df[df['category'] == 'Merch']['revenue'].sum()

# calculating merch share as a % of total revenue
merch_share_pct = (merch_revenue / df['revenue'].sum()) * 100

# printing rounded to 1 decimal place
print(f"Merch Revenue Share: {merch_share_pct:.1f}%")


Merch Revenue Share: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

#Merging df with vendor_name using a left join
joined = pd.merge(
    df,
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one',
)

#identifying the unmatched vendor ID and reporting it
unmatched_vendors = joined[joined['vendor_name'].isna()]['vendor_id'].unique()
print(f"Unmatched Vendor ID: {list(unmatched_vendors)}")

#validating that row count and total revenue didn't change
print(f"Row count unchanged: {len(df)==len(joined)}")
print(f"Revenue total unchanged: {abs(joined['revenue'].sum()-df['revenue'].sum())< 0.01} (${joined['revenue'].sum():,.2f})")


# TODO: merge, validate, and report the unmatched vendor

Unmatched Vendor ID: ['V-18']
Row count unchanged: True
Revenue total unchanged: True ($8,520.00)


**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
#creating a pivot table
pivot_report = pd.pivot_table(
    joined,
    values='revenue',
    index = 'vendor_id',
    columns = 'category',
    aggfunc= 'sum',
    margins=True,
    margins_name='Total'
)

display(pivot_report)

category,Drink,Food,Merch,RainGear,Total
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I would tell these vendors to reallocate inventory away from rain gear and towards food and drinks. Rain gear accounted for just 10.6% of revenues for the four vendors being evaluated, likely because it is contingent on a specific need during rainy weather. Since food and drink is a more generalist need, it should account for more inventory space. Food $4293 of the total $8520 revenues, and would likely do even better with more diverse options and better adverting. Additionally, drink sales can be scaled by having vendors walk up and down the stands to appeal to customers who are thirsty but don't want to leave their seats. This would provide potential for $1554 in total drink revenue to be increased.

b) I think Q5 is the least trustworthy answer because vendor 18 was unmatched. Given that Vendor 18 represents a large % of the overall number of orders (over a quarter), this points to a concerning ratio of unmatched and missing data that gives an unclear picture on the relationships between revenue and order quanitities.